# ЛР-5. Градиентный бустинг

Датасет: UCI HAR (Human Activity Recognition Using Smartphones).  
561 признак, 6 классов активности.  
Реализация: Decision Stumps + Gradient Boosting (OvR).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from mlcore.data.datasets import load_lr5_dataset
from mlcore.preprocessing.scaling import standardize
from mlcore.metrics import accuracy, precision, recall, f1_score, confusion_matrix, roc_curve, auc_score
from mlcore.plotting.evaluation import plot_confusion_matrix, plot_multiclass_roc
from mlcore.utils.timing import timer

ROOT = Path.cwd()
for p in [ROOT, ROOT.parent, ROOT.parent.parent, ROOT.parent.parent.parent]:
    if (p / 'mlcore').exists():
        ROOT = p
        break
ASSETS = ROOT / 'labs/lr-5/assets'
ASSETS.mkdir(parents=True, exist_ok=True)

## 1. Загрузка и анализ данных

In [ ]:
ds = load_lr5_dataset()
# Use canonical train/test split from dataset
X_train_df = ds.raw['x_train']
y_train_df = ds.raw['y_train']
X_test_df = ds.raw['x_test']
y_test_df = ds.raw['y_test']

X_train_raw = X_train_df.values.astype(np.float64)
y_train = y_train_df.values.ravel()
X_test_raw = X_test_df.values.astype(np.float64)
y_test = y_test_df.values.ravel()

ACTIVITY_LABELS = {1: 'WALKING', 2: 'WALK_UP', 3: 'WALK_DOWN', 4: 'SITTING', 5: 'STANDING', 6: 'LAYING'}

print(f'Train: {X_train_raw.shape}, Test: {X_test_raw.shape}')
print(f'Classes: {np.unique(y_train)}')
print(f'Class distribution (train):')
for c in sorted(np.unique(y_train)):
    print(f'  {ACTIVITY_LABELS[c]}: {(y_train == c).sum()}')

In [ ]:
X_train, mean, std = standardize(X_train_raw)
X_test, _, _ = standardize(X_test_raw, mean=mean, std=std)
print(f'Standardized. Train mean~{X_train.mean():.4f}, std~{X_train.std():.4f}')

## 2. Реализация градиентного бустинга

- **Base learner**: Decision Stump (глубина 1)
- **Loss**: log-loss (бинарная кросс-энтропия)
- **Мультикласс**: One-vs-Rest (6 бинарных классификаторов)
- **Оптимизации**: feature subsampling (sqrt(561)≈24), quantile thresholds (max 50)

In [ ]:
class DecisionStump:
    """Depth-1 decision tree for gradient boosting."""

    def __init__(self):
        self.feature_idx = 0
        self.threshold = 0.0
        self.left_value = 0.0
        self.right_value = 0.0

    def fit(self, X: np.ndarray, residuals: np.ndarray,
            feature_subset: np.ndarray | None = None) -> 'DecisionStump':
        n = len(residuals)
        features = feature_subset if feature_subset is not None else np.arange(X.shape[1])
        best_loss = np.inf

        for j in features:
            col = X[:, j]
            # Quantile-based thresholds for speed
            unique_vals = np.unique(col)
            if len(unique_vals) > 50:
                thresholds = np.quantile(col, np.linspace(0.02, 0.98, 50))
            else:
                thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2

            for t in thresholds:
                left_mask = col <= t
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                left_val = residuals[left_mask].mean()
                right_val = residuals[right_mask].mean()
                pred = np.where(left_mask, left_val, right_val)
                loss = np.sum((residuals - pred) ** 2)
                if loss < best_loss:
                    best_loss = loss
                    self.feature_idx = j
                    self.threshold = t
                    self.left_value = left_val
                    self.right_value = right_val
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.where(X[:, self.feature_idx] <= self.threshold,
                        self.left_value, self.right_value)

In [ ]:
class GradientBoostingBinary:
    """Binary gradient boosting classifier (log-loss)."""

    def __init__(self, n_estimators=100, learning_rate=0.1,
                 max_features=24, random_state=42):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_features = max_features
        self.rng = np.random.default_rng(random_state)
        self.stumps: list[DecisionStump] = []
        self.init_value = 0.0
        self.loss_history: list[float] = []

    @staticmethod
    def _sigmoid(z):
        return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))

    def fit(self, X, y):
        p_mean = y.mean()
        self.init_value = np.log(p_mean / (1 - p_mean + 1e-12) + 1e-12)
        F = np.full(len(y), self.init_value)
        n_features = X.shape[1]

        for m in range(self.n_estimators):
            p = self._sigmoid(F)
            residuals = y - p  # negative gradient of log-loss

            feat_subset = self.rng.choice(n_features, size=min(self.max_features, n_features), replace=False)
            stump = DecisionStump().fit(X, residuals, feature_subset=feat_subset)
            self.stumps.append(stump)
            F += self.learning_rate * stump.predict(X)

            eps = 1e-12
            loss = -np.mean(y * np.log(self._sigmoid(F) + eps) + (1 - y) * np.log(1 - self._sigmoid(F) + eps))
            self.loss_history.append(loss)

        return self

    def predict_raw(self, X):
        F = np.full(len(X), self.init_value)
        for stump in self.stumps:
            F += self.learning_rate * stump.predict(X)
        return F

    def predict_proba(self, X):
        return self._sigmoid(self.predict_raw(X))


class GradientBoostingOvR:
    """Multi-class gradient boosting via One-vs-Rest."""

    def __init__(self, **gb_kwargs):
        self.gb_kwargs = gb_kwargs
        self.classifiers: dict[int, GradientBoostingBinary] = {}
        self.classes: np.ndarray = np.array([])

    def fit(self, X, y):
        self.classes = np.unique(y)
        for cls in self.classes:
            print(f'  Training class {cls}...')
            y_binary = (y == cls).astype(np.float64)
            gb = GradientBoostingBinary(**self.gb_kwargs)
            gb.fit(X, y_binary)
            self.classifiers[cls] = gb
        return self

    def predict_proba(self, X):
        proba = np.column_stack([self.classifiers[cls].predict_proba(X) for cls in self.classes])
        proba /= proba.sum(axis=1, keepdims=True)  # normalize
        return proba

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes[proba.argmax(axis=1)]

print('Gradient Boosting classes defined.')

## 3. Обучение

Гиперпараметры:
- `n_estimators=100` — достаточно для HAR (561 сильных признаков)
- `learning_rate=0.1` — стандартный выбор для предотвращения переобучения
- `max_features=24` — sqrt(561), снижает корреляцию между деревьями

In [ ]:
model = GradientBoostingOvR(n_estimators=100, learning_rate=0.1, max_features=24, random_state=42)

with timer('GradientBoostingOvR training'):
    model.fit(X_train, y_train)

In [ ]:
# Training loss curves
fig, ax = plt.subplots(figsize=(8, 4))
for cls in model.classes:
    ax.plot(model.classifiers[cls].loss_history, label=ACTIVITY_LABELS.get(cls, str(cls)), alpha=0.7)
ax.set_xlabel('Iteration')
ax.set_ylabel('Log-loss')
ax.set_title('Training Loss per OvR Classifier')
ax.legend(fontsize='small')
fig.tight_layout()
fig.savefig(ASSETS / 'training_loss.png', dpi=150)
plt.close(fig)
print('Saved training_loss.png')

## 4. Оценка качества

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

metrics = {
    'Accuracy': accuracy(y_test, y_pred),
    'Precision (macro)': precision(y_test, y_pred, average='macro'),
    'Recall (macro)': recall(y_test, y_pred, average='macro'),
    'F1 (macro)': f1_score(y_test, y_pred, average='macro'),
}
for k, v in metrics.items():
    print(f'{k}: {v:.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
class_names = [ACTIVITY_LABELS[c] for c in sorted(np.unique(y_test))]
fig = plot_confusion_matrix(cm, class_names=class_names)
fig.savefig(ASSETS / 'confusion_matrix.png', dpi=150)
plt.close(fig)
print('Saved confusion_matrix.png')

In [ ]:
# Multiclass ROC (One-vs-Rest)
fpr_dict, tpr_dict, auc_dict = {}, {}, {}
for i, cls in enumerate(sorted(np.unique(y_test))):
    y_bin = (y_test == cls).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, y_proba[:, i])
    auc_val = auc_score(fpr, tpr)
    name = ACTIVITY_LABELS[cls]
    fpr_dict[name] = fpr
    tpr_dict[name] = tpr
    auc_dict[name] = auc_val

fig = plot_multiclass_roc(fpr_dict, tpr_dict, auc_dict)
fig.savefig(ASSETS / 'multiclass_roc.png', dpi=150)
plt.close(fig)
print('AUC per class:', {k: f'{v:.3f}' for k, v in auc_dict.items()})
print('Saved multiclass_roc.png')

## 5. Выводы

1. Градиентный бустинг на decision stumps реализован с нуля (numpy).
2. OvR стратегия позволяет использовать бинарный бустинг для 6-классовой задачи.
3. Feature subsampling (24 из 561) ускоряет обучение без существенной потери качества.
4. HAR — хорошо разделимая задача, бустинг достигает высокого accuracy.